# Campaign analysis

Reads the `summary.csv` / `summary.json` that `reduce` already wrote and
recomputes nothing, so this cannot disagree with the artifact it describes.

**On Modal Notebooks:** attach the `rfd3-pipeline` Volume from the Files
panel's filesystem tab. It mounts at `/mnt/rfd3-pipeline` and the next cell
finds it. Run `modal run app.py::publish_tools` once beforehand so
`rfd3_pipeline.analysis` is importable from the volume.

**Locally:** run from the repo and it picks up `src/` instead.

Set the campaign and run below; everything after is read-only.

In [ ]:
CAMPAIGN = "motif_scaffold_alphav_beta3"
RUN = "alphav_beta3_02"

In [ ]:
import sys
from pathlib import Path

# `rfd3_pipeline.analysis` holds every function below, so the notebook and
# `experiments/metrics_table.py` cannot drift. Found in the repo when run
# locally, or on the volume where `publish_tools` puts it.
for candidate in ("src", "../src", "/mnt/rfd3-pipeline/_tools", "/data/_tools"):
    if (Path(candidate) / "rfd3_pipeline" / "analysis.py").is_file():
        sys.path.insert(0, str(Path(candidate).resolve()))
        break

try:
    from rfd3_pipeline import analysis
except ModuleNotFoundError:
    raise SystemExit(
        "rfd3_pipeline not importable. From a checkout, run this notebook with "
        "the repo root as the working directory. On Modal, run "
        "`modal run app.py::publish_tools` once to copy it onto the volume."
    )

run_dir = analysis.find_run(CAMPAIGN, RUN)
records = analysis.load_records(run_dir)
summary = analysis.load_summary(run_dir)
print(f"{run_dir}\n{len(records)} records")

## 0. The funnel

`n_pass` / `n_fail` per stage are the pass rates. `stage_reached` is
deliberately not shown: it counts where records came to rest, so for every
stage but the last those are casualties, not survivors.

In [ ]:
print("\n".join(analysis.funnel(summary)))

## 1. Distributions

What calibrating a threshold needs: the spread, not any one design's value.
Nearest-rank quantiles, so every number printed is one a design actually had.
A threshold is defensible when it sits somewhere defensible here — look for
bimodality, which is what `placement` shows.

In [ ]:
for stage in analysis.STAGES:
    rows = analysis.distribution(records, stage)
    if rows:
        print(analysis.render(rows), end="\n\n")

## 2. What each gate term rejected

A gate whose terms never fail is not gating. `threshold_ipTM` sat at 0.3 for
months while every sequence measured scored 0.735 or better, and nothing said
so. `sole` is the actionable column: a design that failed three terms is not
evidence about any one of them.

In [ ]:
for stage in analysis.STAGES:
    work = analysis.gate_terms(records, stage)
    if not work["n"]:
        continue
    print(f"{stage}: {work['passed']}/{work['n']} passed")
    for metric, count in work["rejected"].items():
        print(f"   {metric:28s} rejected {count}")
    for metric, count in work["sole"].items():
        print(f"   sole reason: {metric:22s} {count}")
    print()

## 3. Holo against CV

**Holo and CV are supposed to disagree.** The agreed set is the campaign's
result precisely because two independent models concurred where they usually do
not — on `alphav_beta3_02`, 16 of 138. Do not read the rest as a yield problem,
and do not try to predict membership from holo: if that worked, CV would be
measuring nothing.

In [ ]:
print(analysis.render(analysis.compare(records)))

agreed = analysis.agreed(records)
folded = [r for r in records if analysis.reached(r, "cv")]
print(f"\nCV confirms holo on {len(agreed)}/{len(folded)} sequences\n")
print(analysis.render(analysis.by_design(agreed)))

## 4. Does ipSAE track placement?

`zeros` counts values of *exactly* 0.0, printed beside the median because a
median of 0.000 hides whether it was half the band or all of it. For ipSAE an
exact 0.0 means no residue pair fell under `ipsae.pae_cutoff` — a verdict about
the interface, not missing data, and continuous with the 0.01 values rather
than a separate failure mode.

At holo this tracks placement cleanly. At CV it does not: OpenFold3's PAE floor
straddles the cutoff, which is why CV ranks on ipSAE and never gates on it.

In [ ]:
for stage in ("holo", "cv"):
    rows = analysis.bucket(
        records, f"{stage}_ipsae_min", f"{stage}_rmsd_binder_on_target"
    )
    if rows:
        print(f"{stage}: ipSAE by placement band")
        print(analysis.render(rows), end="\n\n")

## 5. One sequence, in full

Sequence counts overstate what a campaign found — check `by_design` above
before treating *n* agreed sequences as *n* results. Then read one record
whole: every stage's view of the same design, side by side.

In [ ]:
print(analysis.render(analysis.top(records, "cv_ipsae_min", 10)))

In [ ]:
best = analysis.top(records, "cv_ipsae_min", 1)[0]["seq_id"]
for column, value in analysis.record_for(records, best).items():
    print(f"{column:34s} {str(value)[:70]}")